In [0]:
%run ../_shared/00_project_config

In [0]:
%run ../_shared/01_spark_utils

In [0]:
from pyspark.sql import functions as F


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Project-wide schema names and supported FBref leagues are loaded from
# 00_project_config.
#
# Reusable reading, cleaning, validation, deduplication, and writing helpers
# are loaded from 01_spark_utils.

SILVER_TABLE = f"{SILVER_SCHEMA}.team_miscellaneous_stats"

FBREF_DATASET = "miscellaneous"

# One Silver row represents:
#
#   team + season + league
#
# for the FBref miscellaneous team-statistics dataset.
BUSINESS_KEY = [
    "team_name",
    "season_code",
    "league_code",
]

REQUIRED_SILVER_COLUMNS = [
    "team_name",
    "team_key",
    "season_code",
    "season_start_year",
    "league_code",
]


# -----------------------------------------------------------------------------
# Build expected Bronze table list
# -----------------------------------------------------------------------------
# FBref Bronze maintains one table per:
#
#   dataset + league
#
# Each Bronze table contains all supported seasons for that league.

BRONZE_TABLES = [
    f"{BRONZE_SCHEMA}.fbref_{FBREF_DATASET}_{league_code}"
    for league_code in FBREF_LEAGUES
]


# -----------------------------------------------------------------------------
# Read and combine Bronze tables
# -----------------------------------------------------------------------------
# read_and_union_tables() is strict by default:
#
#   - every expected league table must exist
#   - schemas must be compatible
#
# A missing league should fail the pipeline rather than silently create an
# incomplete Silver dataset.

bronze_df = read_and_union_tables(
    spark,
    BRONZE_TABLES,
)


# -----------------------------------------------------------------------------
# Validate expected FBref source columns
# -----------------------------------------------------------------------------
# FBref is an external HTML source and its structure can change.
# Validate the columns this transformation depends on before selecting them.

REQUIRED_BRONZE_COLUMNS = [
    "Squad",
    "#_Pl",
    "90s",
    "Performance_CrdY",
    "Performance_CrdR",
    "Performance_2CrdY",
    "Performance_Fls",
    "Performance_Fld",
    "Performance_Off",
    "Performance_Crs",
    "Performance_Int",
    "Performance_TklW",
    "Performance_PKwon",
    "Performance_PKcon",
    "Performance_OG",
    "_season",
    "_league",
    "_stat",
    "_ingested_at",
    "_source_file",
    "_source_system",
]

validate_columns_exist(
    bronze_df,
    REQUIRED_BRONZE_COLUMNS,
)


# -----------------------------------------------------------------------------
# Clean top-level strings
# -----------------------------------------------------------------------------
# FBref Bronze is already flat, so the shared top-level string cleaner is
# appropriate here.
#
# It trims string values and converts empty strings to NULL.

bronze_df = clean_top_level_strings(bronze_df)


# -----------------------------------------------------------------------------
# Validate dataset identity
# -----------------------------------------------------------------------------
# All rows processed by this notebook should come from the FBref
# "miscellaneous" dataset.

unexpected_stat_rows = (
    bronze_df
    .filter(
        F.col("_stat").isNull()
        | (F.col("_stat") != F.lit(FBREF_DATASET))
    )
    .limit(1)
    .count()
)

if unexpected_stat_rows > 0:
    raise ValueError(
        f"Unexpected _stat value found while processing {FBREF_DATASET}."
    )


# -----------------------------------------------------------------------------
# Select, rename, and cast Silver columns
# -----------------------------------------------------------------------------
# Source-oriented FBref field names are converted into clear analytical names.
#
# The "90s" field is retained because it provides useful context for the
# aggregate team statistics and can later support per-90 calculations.

silver_df = (
    bronze_df
    .select(

        # ---------------------------------------------------------------------
        # Team
        # ---------------------------------------------------------------------

        clean_string("Squad")
        .alias("team_name"),

        F.col("`#_Pl`")
        .cast("int")
        .alias("players_used_count"),

        F.col("`90s`")
        .cast("double")
        .alias("nineties_played"),


        # ---------------------------------------------------------------------
        # Discipline
        # ---------------------------------------------------------------------

        F.col("Performance_CrdY")
        .cast("int")
        .alias("yellow_cards"),

        F.col("Performance_CrdR")
        .cast("int")
        .alias("red_cards"),

        F.col("Performance_2CrdY")
        .cast("int")
        .alias("second_yellow_cards"),


        # ---------------------------------------------------------------------
        # Fouls and offsides
        # ---------------------------------------------------------------------

        F.col("Performance_Fls")
        .cast("int")
        .alias("fouls_committed"),

        F.col("Performance_Fld")
        .cast("int")
        .alias("fouls_drawn"),

        F.col("Performance_Off")
        .cast("int")
        .alias("offsides"),


        # ---------------------------------------------------------------------
        # Defensive / possession-related activity
        # ---------------------------------------------------------------------

        F.col("Performance_Crs")
        .cast("int")
        .alias("crosses"),

        F.col("Performance_Int")
        .cast("int")
        .alias("interceptions"),

        F.col("Performance_TklW")
        .cast("int")
        .alias("tackles_won"),


        # ---------------------------------------------------------------------
        # Penalties and own goals
        # ---------------------------------------------------------------------

        F.col("Performance_PKwon")
        .cast("int")
        .alias("penalty_kicks_won"),

        F.col("Performance_PKcon")
        .cast("int")
        .alias("penalty_kicks_conceded"),

        F.col("Performance_OG")
        .cast("int")
        .alias("own_goals"),


        # ---------------------------------------------------------------------
        # Season / league context
        # ---------------------------------------------------------------------

        F.col("_season")
        .cast("string")
        .alias("season_code"),

        season_code_to_start_year("_season")
        .alias("season_start_year"),

        F.col("_league")
        .cast("string")
        .alias("league_code"),


        # ---------------------------------------------------------------------
        # Bronze lineage metadata
        # ---------------------------------------------------------------------

        F.col("_ingested_at")
        .cast("timestamp")
        .alias("_bronze_ingested_at"),

        F.col("_source_file")
        .cast("string")
        .alias("_bronze_source_file"),

        F.col("_source_system")
        .cast("string")
        .alias("_source_system"),
    )
)



# -----------------------------------------------------------------------------
# Validate business key before deduplication
# -----------------------------------------------------------------------------
# Validate the complete business key before deduplication so invalid NULL keys
# cannot be grouped together and reduced to one arbitrary record.

validate_required_not_null(
    silver_df,
    BUSINESS_KEY,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Deduplicate repeated Bronze snapshots
# -----------------------------------------------------------------------------
# Bronze currently uses append mode, so rerunning the same FBref Bronze load
# can create multiple copies of the same team/season/league row.
#
# Keep the version from the latest Bronze ingestion.

silver_df = deduplicate_latest(
    silver_df,
    key_columns=BUSINESS_KEY,
    order_column="_bronze_ingested_at",
    tie_breaker_columns=[
        "_bronze_source_file",
    ],
)


# -----------------------------------------------------------------------------
# Resolve canonical team key
# -----------------------------------------------------------------------------

team_mappings_df = load_team_mappings_by_name(
    spark,
    SILVER_SCHEMA,
    "fbref",
)

silver_df = resolve_team_keys_by_name(
    silver_df,
    team_mappings_df,
    source_name_column="team_name",
    output_key_column="team_key",
    fail_on_unmapped=True,
)

print("FBref canonical team resolution completed.")

# -----------------------------------------------------------------------------
# Add Silver processing metadata
# -----------------------------------------------------------------------------

silver_df = silver_df.withColumn(
    "_silver_processed_at",
    F.current_timestamp(),
)


# -----------------------------------------------------------------------------
# Final Silver validation
# -----------------------------------------------------------------------------
# Required identifiers are validated as a final data-quality gate.
#
# The helper also returns the final row count, avoiding an additional count()
# action.

validation_metrics = validate_required_not_null(
    silver_df,
    REQUIRED_SILVER_COLUMNS,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Write Silver table
# -----------------------------------------------------------------------------
# The current project phase uses full-refresh Silver processing.

write_full_refresh(
    silver_df,
    SILVER_TABLE,
)


# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {validation_metrics['row_count']} "
    f"team miscellaneous-stat records into {SILVER_TABLE}."
)